# Kunitz HMM evaluation reanalysis
Elham Hadadi — Laboratory of Bioinformatics I

This notebook documents the retrospective expansion of the negative evaluation background. The structural HMM and original positive allocation are unchanged. The original sampled-negative benchmark is retained as historical evidence. A new cutoff is selected using expanded validation data and is then applied to the corresponding test data.

The protocol was fixed before computing the reanalysis metrics, but the original project results had already been inspected. This is a retrospective reanalysis, not a fresh external test. It reparses archived outputs; it does not retrain or rerun HMMER, or execute programs from the archive.


In [1]:
from pathlib import Path
import hashlib,json,zipfile
import sys
archive_name='kunitz_project_backup_20261107T213430_789701Z.zip'
archive=Path('/content/drive/MyDrive/kunitz_project_backups')/archive_name if 'google.colab' in sys.modules else Path.home()/'Downloads'/archive_name
if not archive.is_file() and 'google.colab' in sys.modules:
 from google.colab import drive
 drive.mount('/content/drive')
if not archive.is_file():
 raise FileNotFoundError('Place the original project backup in Downloads, or mount its existing Drive folder in Colab.')
out=Path('work/raw_audit');out.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
 manifest=json.loads(z.read('backup_file_manifest.json'))
 entries={r['relative_path']:r for r in manifest['files']}
 selected=[]
 for rel,r in entries.items():
  keep=(rel.startswith('data/processed/') and r['size_bytes']<20_000_000) or (rel.startswith(('results/test/','results/validation/','results/hmm/','results/robustness/','results/similarity/','results/swissprot_scan/')) and ('20261107' in rel or '/current_' in rel) and not rel.endswith(('.png','.pdf')))
  if keep and ('/20261107T205124_' not in rel) and not rel.endswith('hmmsearch_report.txt'):
   target=out/rel
   assert target.resolve().is_relative_to(out.resolve())
   data=z.read('kunitz_project/'+rel)
   assert len(data)==r['size_bytes'] and hashlib.sha256(data).hexdigest()==r['sha256'],rel
   target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data);selected.append(rel)
 (out/'extraction_manifest.json').write_text(json.dumps({'archive':archive.name,'archive_sha256':hashlib.file_digest(archive.open('rb'),'sha256').hexdigest(),'verified_files':selected},indent=2))
print('Verified and extracted',len(selected),'data files; no archived code executed.')

# Verify the two larger source inputs needed by the complete-background analysis.
with zipfile.ZipFile(archive) as z:
 for rel in ['data/processed/SwissProt_negative_candidate_metadata.csv','data/raw/uniprot/swissprot_complete_2026_03/swissprot_complete.fasta']:
  entry=entries[rel];target=out/rel
  data=z.read('kunitz_project/'+rel)
  assert hashlib.sha256(data).hexdigest()==entry['sha256']
  target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)


Verified and extracted 236 data files; no archived code executed.


## Fixed reanalysis protocol
All nonfragment reference negatives are included. Original positive groups and negative assignments are retained. Exact complete-sequence duplicates are grouped globally; new negative groups are assigned deterministically using seed 42 and accession hashing. No new exhaustive homology clustering is claimed. The selection criterion is validation MCC, with the original midpoint tie rule. The test cutoff is not adjusted afterwards.


In [2]:
Path('work/expanded_design.json').write_text('{\n "purpose":"Retrospective reanalysis to assess negative-sampling dependence; not to target a lower sensitivity",\n "model":"Original frozen 53-state structural HMM, unchanged",\n "reference":"Same Swiss-Prot 2026_03 reviewed nonfragment PF00014 definition",\n "positives":"Keep original 174 validation and 174 test positives and their original group allocation",\n "negatives":"All 566072 eligible reference-negative proteins; exact complete-sequence duplicates grouped globally; original negative 80-percent groups also kept intact",\n "negative_assignment":"Keep original negative groups on their original side; for each new connected group, deterministic SHA256 of seed 42 plus smallest accession; even parity validation, odd test",\n "conflicting_labels":"Exclude any complete-sequence group with both reference labels, report count",\n "threshold":"Maximize validation MCC over distinct best-domain scores; midpoint of widest finite optimal interval, higher midpoint on tie; freeze before evaluating test",\n "scores":"Reparse archived raw tblout; use full-scan scores for all proteins and verify original validation/test scores unchanged at decision-relevant scores; inspect any discrepancy",\n "no_hit":"Negative at finite cutoff",\n "limits":"Post hoc; original positive homology limits remain; new negative homology clustering beyond exact duplicates is not performed; annotation labels are operational",\n "iterations":"One fixed seed and design; no retries chosen by resulting test performance"\n}\n')
print(Path('work/expanded_design.json').read_text())


{
 "purpose":"Retrospective reanalysis to assess negative-sampling dependence; not to target a lower sensitivity",
 "model":"Original frozen 53-state structural HMM, unchanged",
 "reference":"Same Swiss-Prot 2026_03 reviewed nonfragment PF00014 definition",
 "positives":"Keep original 174 validation and 174 test positives and their original group allocation",
 "negatives":"All 566072 eligible reference-negative proteins; exact complete-sequence duplicates grouped globally; original negative 80-percent groups also kept intact",
 "negative_assignment":"Keep original negative groups on their original side; for each new connected group, deterministic SHA256 of seed 42 plus smallest accession; even parity validation, odd test",
 "conflicting_labels":"Exclude any complete-sequence group with both reference labels, report count",
 "threshold":"Maximize validation MCC over distinct best-domain scores; midpoint of widest finite optimal interval, higher midpoint on tie; freeze before evaluating 

In [3]:
import csv,json,math,hashlib
from pathlib import Path
root=Path('work/raw_audit')
out=Path('work/raw_recomputed');out.mkdir(exist_ok=True)
def read(rel):
 with (root/rel).open() as f:return list(csv.DictReader(f))
def j(rel):return json.loads((root/rel).read_text())
def rel(p):return p.removeprefix('/content/kunitz_project/')
def metrics(rows,t):
 tp=tn=fp=fn=0
 for row in rows:
  y=int(row['reference_label']);s=row['raw_best_domain_score'];pred=s is not None and s>=t
  if y and pred:tp+=1
  elif y:fn+=1
  elif pred:fp+=1
  else:tn+=1
 den=math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
 return dict(TP=tp,TN=tn,FP=fp,FN=fn,sensitivity=tp/(tp+fn) if tp+fn else None,specificity=tn/(tn+fp) if tn+fp else None,precision=tp/(tp+fp) if tp+fp else None,MCC=(tp*tn-fp*fn)/den if den else None)
allrows={};audit={}; hashes={}
for split,stamp in [('validation','20261107T212941_588852Z'),('test','20261107T212942_643707Z')]:
 dr=f'results/{split}/{stamp}'
 meta=read(f'data/processed/evaluation_split/{split}_metadata.csv')
 saved=read(f'{dr}/{split}_scores.csv')
 sd={r['Entry']:r for r in saved}
 hits={}
 for line in (root/f'{dr}/{split}_hits.tbl').read_text().splitlines():
  if not line or line.startswith('#'):continue
  f=line.split(); assert f[0] not in hits;hits[f[0]]=float(f[8])
 assert set(hits)<=set(sd)
 assert len(meta)==len(saved)==1914
 assert {r['Entry'] for r in meta}==set(sd)
 for row in meta:
  entry=row['Entry'];score=hits.get(entry);row['raw_best_domain_score']=score
  assert int(row['reference_label'])==int(sd[entry]['reference_label'])
  assert row['Sequence']==sd[entry].get('Sequence',row['Sequence'])
  sv=sd[entry]['best_domain_score_bits'];assert (score is None and not sv) or (score is not None and float(sv)==score)
 allrows[split]=meta
 audit[split]={'rows':len(meta),'raw_hits':len(hits),'missing_positive_hits':sum(int(r['reference_label'])==1 and r['raw_best_domain_score'] is None for r in meta)}
 record=j(f'{dr}/run_record.json');hashes[split]=record['model_sha256']
# Independently optimize over finite raw validation scores using midpoints.
v=allrows['validation'];unique=sorted({r['raw_best_domain_score'] for r in v if r['raw_best_domain_score'] is not None})
sweep=[]
for lo,hi in zip(unique[:-1],unique[1:]):
 m=metrics(v,(lo+hi)/2);sweep.append((m['MCC'],lo,hi))
best=max(x[0] for x in sweep);opt=[x for x in sweep if x[0]==best]
lo,hi=max([(x[1],x[2]) for x in opt],key=lambda x:(x[1]-x[0],(x[0]+x[1])/2))
t=(lo+hi)/2
assert math.isclose(t,8.7) and hashes['test']==hashes['validation']
for split,rows in allrows.items():
 audit[split]['recomputed_metrics']=metrics(rows,t)
 scores={label:[r['raw_best_domain_score'] for r in rows if int(r['reference_label'])==label and r['raw_best_domain_score'] is not None] for label in [0,1]}
 audit[split]['lowest_positive']=min(scores[1]);audit[split]['highest_reported_negative']=max(scores[0])
# Independently check frozen membership and overlap from complete sequences.
v=allrows['validation'];trows=allrows['test']
for column in ['Entry','Sequence','similarity_group']:
 assert not ({r[column] for r in v}&{r[column] for r in trows}),column
# Verify exclusions and membership of named difficult examples.
positives=read('data/processed/SwissProt_positive_candidates.csv')
eligible=read('data/processed/SwissProt_eligible_positives.csv')
assert len(positives)==368 and len(eligible)==348
assert {r['Entry'] for r in eligible}=={r['Entry'] for r in v+trows if int(r['reference_label'])==1}
training_audit=j('results/robustness/current_training_audit.json')
training=read(rel(training_audit['audit_path']));print('training fields',list(training[0]))
# Recompute expanded-background check from actual membership tables.
neg=read('data/processed/SwissProt_negative_candidate_metadata.csv')
negids={r['Entry'] for r in neg};assert len(neg)==len(negids)==566072
scan='results/swissprot_scan/20261107T212956_904384Z'
preds=read(scan+'/classification/SwissProt_predicted_Kunitz.csv')
meta=read(scan+'/distribution/metadata_corrected_20261107T213427_876118Z/predicted_protein_distribution_metadata.csv')
md={r['Entry']:r for r in meta}; assert len(md)==426
old_neg={r['Entry'] for r in v+trows if int(r['reference_label'])==0}
expanded=negids-old_neg
fpids={r['Entry'] for r in preds if r['Entry'] in expanded}
assert len(expanded)==562592 and len(fpids)==15
# Reconstruct independently, without using stored predicted labels.
scanraw={}
with (root/scan/'swissprot_hits.tbl').open() as f:
 for line in f:
  if line.startswith('#') or not line.strip():continue
  x=line.split();accession=x[0].split('|')[1] if x[0].startswith(('sp|','tr|')) else x[0]
  assert accession not in scanraw
  scanraw[accession]=float(x[8])
raw_fpids={x for x in expanded if scanraw.get(x,-math.inf)>=8.7}
assert raw_fpids==fpids
rawexpanded=[dict(reference_label='0',raw_best_domain_score=scanraw.get(x)) for x in expanded]
expandedmetric=metrics([r for r in trows if int(r['reference_label'])==1]+rawexpanded,8.7)
results={'status':'Recomputed from verified extracted raw HMMER tblout and metadata; HMMER itself was not rerun','threshold_recomputed_from_validation':t,'optimal_interval':{'lower_exclusive':lo,'upper_inclusive':hi},'same_model_hash_validation_test':hashes['test'],'no_shared_accessions_sequences_or_80pct_groups':True,'splits':audit,'expanded_background_audit':expandedmetric,'expanded_negative_count':len(expanded),'expanded_FP_accessions':sorted(fpids),'positive_exclusions_total':20,'positive_exclusion_criterion':'training overlap or close training similarity, not HMM scores'}
(out/'raw_recalculation.json').write_text(json.dumps(results,indent=2))
with (out/'test_positive_raw_scores.csv').open('w') as f:
 w=csv.DictWriter(f,fieldnames=['Entry','similarity_group','raw_best_domain_score']);w.writeheader();w.writerows({k:r[k] for k in w.fieldnames} for r in trows if int(r['reference_label'])==1)
print(json.dumps(results,indent=2))


training fields ['Entry', 'split', 'training_seed_id', 'alignment_score', 'identity_percent', 'seed_coverage_percent', 'coverage_qualified', 'alignment_columns', 'paired_positions', 'target_start', 'target_end', 'has_coverage_qualified_match', 'best_score_seed_id', 'best_score_identity_percent', 'best_score_seed_coverage_percent', 'training_identity_band']
{
  "status": "Recomputed from verified extracted raw HMMER tblout and metadata; HMMER itself was not rerun",
  "threshold_recomputed_from_validation": 8.7,
  "optimal_interval": {
    "lower_exclusive": 6.1,
    "upper_inclusive": 11.3
  },
  "same_model_hash_validation_test": "3d315c6484a3ad992f3b15d52d881da790dffcb5f010139c8fb73aeca9cb0308",
  "no_shared_accessions_sequences_or_80pct_groups": true,
  "splits": {
    "validation": {
      "rows": 1914,
      "raw_hits": 1001,
      "missing_positive_hits": 0,
      "recomputed_metrics": {
        "TP": 174,
        "TN": 1740,
        "FP": 0,
        "FN": 0,
        "sensitivity"

In [4]:
import csv,json,math,hashlib
from collections import defaultdict,Counter
from pathlib import Path
R=Path('work/raw_audit');O=Path('work/reanalysis');O.mkdir(exist_ok=True)
design=json.loads(Path('work/expanded_design.json').read_text())
(O/'design.json').write_text(json.dumps(design,indent=2))
def read(p):
 with (R/p).open() as f:return list(csv.DictReader(f))
assignment=read('data/processed/evaluation_split/split_assignments.csv');original={r['Entry']:r for r in assignment}
neg=read('data/processed/SwissProt_negative_candidate_metadata.csv');pos=read('data/processed/SwissProt_eligible_positives.csv')
labels={r['Entry']:0 for r in neg};labels.update({r['Entry']:1 for r in pos});assert len(labels)==566420
parent={x:x for x in labels}
def find(x):
 while x!=parent[x]:parent[x]=parent[parent[x]];x=parent[x]
 return x
def union(a,b):
 a,b=find(a),find(b)
 if a!=b:parent[max(a,b)]=min(a,b)
oldgroups={}
for r in assignment:
 g=r['similarity_group'];x=r['Entry']
 if g in oldgroups:union(x,oldgroups[g])
 else:oldgroups[g]=x
seq_first={};seen=set();seqhash={}
fasta=R/'data/raw/uniprot/swissprot_complete_2026_03/swissprot_complete.fasta'
def consume(header,lines):
 if not header:return
 x=header.split()[0].split('|')[1];
 if x not in labels:return
 sequence=''.join(lines);h=hashlib.sha256(sequence.encode()).hexdigest();seqhash[x]=h;seen.add(x)
 if h in seq_first:union(x,seq_first[h])
 else:seq_first[h]=x
header=None;lines=[]
with fasta.open() as f:
 for line in f:
  if line.startswith('>'):consume(header,lines);header=line[1:].strip();lines=[]
  else:lines.append(line.strip())
consume(header,lines);assert seen==set(labels)
groups=defaultdict(list)
for x in labels:groups[find(x)].append(x)
alloc={};conflicts=[]
for root,members in groups.items():
 ys={labels[x] for x in members}
 if len(ys)>1:conflicts.extend(members);continue
 old={original[x]['split'] for x in members if x in original}
 assert len(old)<=1
 side=next(iter(old)) if old else ('validation' if int(hashlib.sha256(('42:'+min(members)).encode()).hexdigest(),16)%2==0 else 'test')
 for x in members:alloc[x]=side
# Persist the split before reading scores.
with (O/'assignments.csv').open('w') as f:
 w=csv.writer(f);w.writerow(['Entry','reference_label','split','group','sequence_sha256'])
 for x in sorted(alloc):w.writerow([x,labels[x],alloc[x],find(x),seqhash[x]])
print('Frozen new split:',dict(Counter((alloc[x],labels[x]) for x in alloc)),'conflicts',len(conflicts),flush=True)
# Read only fixed scoring outputs; retain all no-hit proteins.
scores={}
scan='results/swissprot_scan/20261107T212956_904384Z/swissprot_hits.tbl'
with (R/scan).open() as f:
 for line in f:
  if line.startswith('#') or not line.strip():continue
  z=line.split();x=z[0].split('|')[1] if z[0].startswith(('sp|','tr|')) else z[0];scores[x]=float(z[8])
diffs=[]
for split,stamp in [('validation','20261107T212941_588852Z'),('test','20261107T212942_643707Z')]:
 old=read(f'results/{split}/{stamp}/{split}_scores.csv')
 for r in old:
  x=r['Entry'];a=float(r['best_domain_score_bits']) if r['best_domain_score_bits'] else -math.inf;b=scores.get(x,-math.inf)
  if a!=b:diffs.append({'Entry':x,'label':int(r['reference_label']),'old':a,'full':b})
print('Score discrepancies',len(diffs),diffs[:10],flush=True)
# Database E-value reporting may add extremely weak hits. All positive scores
# must agree; any changed negative must remain below the original cutoff.
assert all(d['label']==0 and max(d['old'],d['full'])<8.7 for d in diffs)
ids={side:sorted(x for x in alloc if alloc[x]==side) for side in ['validation','test']}
def metric(side,threshold):
 count=Counter()
 for x in ids[side]:
  p=scores.get(x,-math.inf)>=threshold;y=labels[x]
  count['TP' if y and p else 'FN' if y else 'FP' if p else 'TN']+=1
 tp,fn,fp,tn=[count[x] for x in ['TP','FN','FP','TN']]
 den=math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
 return dict(TP=tp,FN=fn,FP=fp,TN=tn,sensitivity=tp/(tp+fn),specificity=tn/(tn+fp),precision=tp/(tp+fp) if tp+fp else 0,MCC=(tp*tn-fp*fn)/den if den else 0,F1=2*tp/(2*tp+fp+fn))
# Efficient cumulative validation sweep over unique score bins.
bins=defaultdict(lambda:[0,0]);total=Counter(labels[x] for x in ids['validation'])
for x in ids['validation']:
 s=scores.get(x,-math.inf)
 if math.isfinite(s):bins[s][labels[x]]+=1
values=sorted(bins);tp=total[1];fp=total[0]
# Remove missing hits, then move each finite bin below cutoff.
missing=Counter(labels[x] for x in ids['validation'] if x not in scores);tp-=missing[1];fp-=missing[0]
sweep=[]
for lo,hi in zip(values[:-1],values[1:]):
 fp-=bins[lo][0];tp-=bins[lo][1];fn=total[1]-tp;tn=total[0]-fp
 den=math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn));m=(tp*tn-fp*fn)/den if den else 0
 sweep.append(dict(lower=lo,upper=hi,MCC=m,TP=tp,FP=fp,FN=fn,TN=tn))
best=max(r['MCC'] for r in sweep);optimal=[r for r in sweep if abs(r['MCC']-best)<1e-12]
# Merge adjoining tied intervals, as specified before scoring.
blocks=[]
for r in optimal:
 if blocks and blocks[-1][1]==r['lower']:blocks[-1][1]=r['upper']
 else:blocks.append([r['lower'],r['upper']])
lo,hi=max(blocks,key=lambda v:(v[1]-v[0],(v[0]+v[1])/2));threshold=(lo+hi)/2
selected={'threshold_bits':threshold,'optimal_interval_lower_exclusive':lo,'optimal_interval_upper_inclusive':hi,'metric':'MCC','design_sha256':hashlib.sha256(Path('work/expanded_design.json').read_bytes()).hexdigest(),'split_sha256':hashlib.sha256((O/'assignments.csv').read_bytes()).hexdigest(),'validation_metrics':metric('validation',threshold),'test_metrics_not_used':True}
(O/'selected_threshold.json').write_text(json.dumps(selected,indent=2))
# Evaluate test only after the threshold record has been written.
result=dict(status='Retrospective expanded-negative reanalysis of archived scores; fixed HMM',selected_threshold=selected,test_metrics=metric('test',threshold),counts={side:{str(y):sum(labels[x]==y for x in ids[side]) for y in [0,1]} for side in ids},label_conflicts_excluded=conflicts,weak_negative_score_discrepancies=diffs,original_cutoff_metrics={side:metric(side,8.7) for side in ids})
with (O/'test_predictions.csv').open('w') as f:
 w=csv.writer(f);w.writerow(['Entry','reference_label','score_bits','predicted_label','outcome'])
 for x in ids['test']:
  s=scores.get(x,-math.inf);p=s>=threshold;y=labels[x]
  w.writerow([x,y,s if math.isfinite(s) else '',int(p),'TP' if y and p else 'FN' if y else 'FP' if p else 'TN'])
with (O/'threshold_sweep.csv').open('w') as f:
 w=csv.DictWriter(f,fieldnames=list(sweep[0]));w.writeheader();w.writerows(sweep)
(O/'results.json').write_text(json.dumps(result,indent=2))
print(json.dumps({k:v for k,v in result.items() if k!='weak_negative_score_discrepancies'},indent=2))


Frozen new split: {('test', 0): 283818, ('validation', 0): 282254, ('test', 1): 174, ('validation', 1): 174} conflicts 0
Score discrepancies 0 []
{
  "status": "Retrospective expanded-negative reanalysis of archived scores; fixed HMM",
  "selected_threshold": {
    "threshold_bits": 17.2,
    "optimal_interval_lower_exclusive": 13.6,
    "optimal_interval_upper_inclusive": 20.8,
    "metric": "MCC",
    "design_sha256": "fa5955de94a52e2281d7bd2a88523b3a9b11999972489e64c0aab558f92a0a7d",
    "split_sha256": "849fc114829003a5ee2f0064352289b59d79355cbb52ae5a489c3ab78152e133",
    "validation_metrics": {
      "TP": 173,
      "FN": 1,
      "FP": 0,
      "TN": 282254,
      "sensitivity": 0.9942528735632183,
      "specificity": 1.0,
      "precision": 1.0,
      "MCC": 0.9971205298403392,
      "F1": 0.9971181556195965
    },
    "test_metrics_not_used": true
  },
  "test_metrics": {
    "TP": 172,
    "FN": 2,
    "FP": 2,
    "TN": 283816,
    "sensitivity": 0.9885057471264368,
    "s

In [5]:
import csv,json,math
from pathlib import Path
R=Path('work/raw_audit');O=Path('work/reanalysis')
result=json.loads((O/'results.json').read_text());t=result['selected_threshold']['threshold_bits']
scan=R/'results/swissprot_scan/20261107T212956_904384Z'
def read(p):
 with p.open() as f:return list(csv.DictReader(f))
oldpred=read(scan/'classification/SwissProt_predicted_Kunitz.csv');oldpos={r['Entry'] for r in read(scan/'annotation_comparison/predicted_with_PF00014.csv')}
meta=read(scan/'distribution/summary/predictions_with_taxonomic_bins.csv');md={r['Entry']:r for r in meta}
scores={}
with (scan/'swissprot_hits.tbl').open() as f:
 for line in f:
  if not line.strip() or line.startswith('#'):continue
  x=line.split();entry=x[0].split('|')[1] if x[0].startswith(('sp|','tr|')) else x[0];scores[entry]=float(x[8])
newids={x for x,s in scores.items() if s>=t};assert newids<={r['Entry'] for r in oldpred}
domcount={x:0 for x in newids};domains=[];oldcount=0
with (scan/'swissprot_domains.domtbl').open() as f:
 for line in f:
  if not line.strip() or line.startswith('#'):continue
  x=line.split();entry=x[0].split('|')[1] if x[0].startswith(('sp|','tr|')) else x[0];s=float(x[13]);oldcount+=s>=8.7
  if s>=t:
   assert entry in newids;domcount[entry]+=1;domains.append(dict(Entry=entry,score=s,hmm_start=int(x[15]),hmm_end=int(x[16]),ali_start=int(x[17]),ali_end=int(x[18])))
assert oldcount==514 and all(n>0 for n in domcount.values())
miss=oldpos-newids;extra=newids-oldpos
review=read(scan/'discordant_domain_review/interpretation/discordant_prediction_interpretation.csv');rv={r['Entry']:r for r in review}
from collections import Counter
stats={'threshold_bits':t,'database_proteins':575748,'predicted_proteins':len(newids),'PF00014_detected':len(oldpos&newids),'PF00014_missed':len(miss),'additional_predictions':len(extra),'annotation_agreement':len(oldpos&newids)/len(newids),'domains_above_threshold':len(domains),'multidomain_proteins':sum(n>1 for n in domcount.values()),'review_counts':dict(Counter(rv[x]['review_category'] for x in extra)),'taxonomic_counts':dict(Counter(md[x]['taxonomic_bin'] for x in newids)),'explicit_fragment_count':sum(md[x]['is_explicit_fragment'].lower()=='true' for x in newids),'missed_PF00014':[dict(Entry=x,score=scores.get(x),is_fragment=md[x]['is_explicit_fragment'],length=md[x].get('Length'),taxon=md[x]['taxonomic_bin']) for x in sorted(miss)],'additional_accessions':sorted(extra)}
errors=[r for r in read(O/'test_predictions.csv') if r['outcome'] in ['FP','FN']]
for r in errors:
 if r['Entry'] in rv:r['annotation_review']=rv[r['Entry']]['review_category']
stats['test_errors']=errors
(O/'full_scan_reanalysis.json').write_text(json.dumps(stats,indent=2))
for name,rows in [('full_scan_predictions.csv',[dict(Entry=x,score_bits=scores[x],PF00014=int(x in oldpos),domains=domcount[x]) for x in sorted(newids)]),('test_errors.csv',errors)]:
 with (O/name).open('w') as f:
  w=csv.DictWriter(f,fieldnames=list(dict.fromkeys(k for r in rows for k in r)));w.writeheader();w.writerows(rows)
print(json.dumps(stats,indent=2))


{
  "threshold_bits": 17.2,
  "database_proteins": 575748,
  "predicted_proteins": 398,
  "PF00014_detected": 395,
  "PF00014_missed": 3,
  "additional_predictions": 3,
  "annotation_agreement": 0.992462311557789,
  "domains_above_threshold": 485,
  "multidomain_proteins": 52,
  "review_counts": {
    "Kunitz_supported_other": 2,
    "Kunitz_supported_fragment": 1
  },
  "taxonomic_counts": {
    "Vertebrates": 223,
    "Arthropods": 97,
    "Other metazoans": 17,
    "Cnidarians": 57,
    "Nematodes": 4
  },
  "explicit_fragment_count": 31,
  "missed_PF00014": [
    {
      "Entry": "A0A1Q1NL17",
      "score": 11.2,
      "is_fragment": "False",
      "length": "101",
      "taxon": "Arthropods"
    },
    {
      "Entry": "D3GGZ8",
      "score": 11.3,
      "is_fragment": "False",
      "length": "195",
      "taxon": "Nematodes"
    },
    {
      "Entry": "O62247",
      "score": 13.9,
      "is_fragment": "False",
      "length": "202",
      "taxon": "Nematodes"
    }
  ],
  "a

In [6]:
import csv,json,math,hashlib
from pathlib import Path
O=Path('work/reanalysis');R=Path('work/raw_audit')
r=json.loads((O/'results.json').read_text());t=r['selected_threshold']['threshold_bits'];assert t==17.2
with (O/'test_predictions.csv').open() as f:rows=list(csv.DictReader(f))
from collections import Counter
c=Counter(x['outcome'] for x in rows);assert c=={'TP':172,'FN':2,'FP':2,'TN':283816}
for x in rows:
 score=float(x['score_bits']) if x['score_bits'] else -math.inf
 assert int(x['predicted_label'])==int(score>=t)
assert len(rows)==283992
with (O/'assignments.csv').open() as f:a=list(csv.DictReader(f))
assert len(a)==566420 and len({x['Entry'] for x in a})==566420
for column in ['Entry','group','sequence_sha256']:
 v={x[column] for x in a if x['split']=='validation'};s={x[column] for x in a if x['split']=='test'}
 assert not(v&s)
f=json.loads((O/'full_scan_reanalysis.json').read_text());assert f['predicted_proteins']==f['PF00014_detected']+f['additional_predictions']==398
assert f['PF00014_detected']+f['PF00014_missed']==398
assert sum(f['taxonomic_counts'].values())==398
with (R/'results/robustness/training_audit_20261107T213423_499191Z/positive_training_similarity_audit.csv').open() as h:tr={x['Entry']:x for x in csv.DictReader(h)}
remote=[x for x in rows if x['reference_label']=='1' and float(tr[x['Entry']]['identity_percent'])<40]
assert len(remote)==9 and sum(x['predicted_label']=='1' for x in remote)==7
print('PASS: all reanalysis predictions match the frozen threshold; counts and full-scan summaries reconcile.')
print('PASS: no shared accessions, connected groups or exact complete-sequence hashes across the new split.')
print('PASS: 7/9 test positives below 40% training identity detected at 17.2 bits.')
print('Scope: archived raw scores were reprocessed; HMMER was not rerun, and distant homology is not excluded.')


PASS: all reanalysis predictions match the frozen threshold; counts and full-scan summaries reconcile.
PASS: no shared accessions, connected groups or exact complete-sequence hashes across the new split.
PASS: 7/9 test positives below 40% training identity detected at 17.2 bits.
Scope: archived raw scores were reprocessed; HMMER was not rerun, and distant homology is not excluded.


## Interpretation and limits
The original 8.7-bit result was arithmetically correct. Expanding the validation background reveals rare annotation-defined false positives and changes the MCC-optimal cutoff to 17.2 bits. The corresponding test has TP=172, FN=2, FP=2 and TN=283,816: sensitivity and precision are 98.85%, MCC is 0.9885.

The two test false positives lack PF00014 but have other archived Kunitz evidence, so annotation disagreement must not be equated with proven biological error. Original positive homology limitations remain. Added negatives are grouped for exact duplication and original accepted groups, not for all evolutionary relationships. This notebook cannot establish performance on an unseen external database.

The complete-database application at 17.2 bits yields 398 candidates, comprising 395 PF00014-annotated proteins and three additional predictions. Three PF00014-annotated nonfragments are missed. The full scan includes training and evaluation proteins.

At the revised cutoff, seven of the nine test positives below 40% training identity are detected; the small subgroup illustrates the sensitivity trade-off.
